# Main for Energy Simulation 

Simulates the energy consumption for each track, calculates the SoC each truck would have and calculates the resulting load profiles for the freight forwarding locations 

This notebook is to be executed after data/data_handling/data-aquisition.ipynb and main_data_analysis.ipynb

### Imports 

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import seaborn as sns
import sys
import matplotlib.pyplot as plt 
import matplotlib.patches as mpatches 
import matplotlib.lines as mlines
import matplotlib.colors
import matplotlib.ticker as mticker

from utils.style_config import *

sys.path.append('energy_sim')
import data_handling.data_processing as dp
import energy_sim.sequential_analysis as sq
import energy_sim.load_profile as lp

### Style parameters

In [ ]:
plt.rcParams.update({
    'text.usetex': True,
    'font.family': 'sans-serif',
    'font.sans-serif': 'Arial',
    'font.size': 12,
    'text.latex.preamble': r'\usepackage{siunitx}'
})

### Scenario Parameters

In [ ]:
# Constant charging powers 
params_set = pd.DataFrame(
   {'default':{'batt_cap': 572,
         'dispatch': 'constant',
         'soc_min': 0.15,
         'soc_max': 0.9,
         'charging_powers': {'home base': 150, 'industrial area': 0},
         },
          'big_batt':{'batt_cap': 798,
         'dispatch': 'constant',
         'soc_min': 0.15,
         'soc_max': 0.9,
         'charging_powers': {'home base': 150, 'industrial area': 0},
         },
          'destination':{'batt_cap': 572,
         'dispatch': 'constant',
         'soc_min': 0.15,
         'soc_max': 0.9,
         'charging_powers': {'home base': 150, 'industrial area': 350},
         },
          'low_home':{'batt_cap': 572,
         'dispatch': 'constant',
         'soc_min': 0.15,
         'soc_max': 0.9,
         'charging_powers': {'home base': 50, 'industrial area': 0},
         },
          'high_home':{'batt_cap': 572,
         'dispatch': 'constant',
         'soc_min': 0.15,
         'soc_max': 0.9,
         'charging_powers': {'home base': 350, 'industrial area': 0},
         },
          'ultra_home':{'batt_cap': 572,
         'dispatch': 'constant',
         'soc_min': 0.15,
         'soc_max': 0.9,
         'charging_powers': {'home base': 10000, 'industrial area': 0},
         },
}
)
display(params_set)

# set one default scenario
DEFAULT_SCENARIO = 'default'

## Perform Energy Simulation based on BET.OS script 

The energy simulation takes very long to run (in total approx. 24 h) and should just only be executed once 

First run the faulty_tracks() function to identify tracks with missing data (results will be saved in mismatched_tracks.csv)
Then run run_energy_sim() to calculate energy consumption for each track (results will be saved in tracks_filtered_with_energy.csv)
Run run_energy_sim() will additionally create zero_speed_tracks.csv which lists tracks that have no speed data

The outputs are saved as csvs

### Run once:

In [ ]:
#faulty_tracks()

# trips_energy = run_energy_sim(override_mismatched_tracks=False)

# "Can either pass on trips_energy directly from the simulation or let the function read in tracks_with_energy_raw.csv"
# df_trip_energy = clean_energy_sim_data(trips=None)

### After running the energy simulation once, read the csv instead of excecuting the simulation

In [ ]:
df_trips = pd.read_csv('input/stations/tracks_filtered.csv', index_col='track_id')
df_trip_energy= pd.read_csv('output/csvs/tracks_with_energy.csv', index_col='track_id')
if False: # debugging
    df_trip_energy = df_trip_energy.head(10000)
    df_trips = df_trips.loc[df_trips.index.to_series().isin(df_trip_energy.index)]

df_trips['stop_time'] = pd.to_datetime(df_trips['stop_time'], format='ISO8601', utc=True)
df_trips['start_time'] = pd.to_datetime(df_trips['start_time'], format='ISO8601', utc=True)
df_stops = dp.process_stops_data(df_trips.drop(columns=['duration', 'max_speed_kmh', 'avg_speed_kmh', 'gross_vehicle_weight', 'total_mass_with_trailer', 'axle_class',
                       "avg_speed", "max_speed", "n_signal_loss", "d_signal_loss", "r_signal_loss", "avg_hdop"]))
df_occ_energy = sq.combine_tracks_and_stops(df_stops = df_stops, df_tracks_with_energy = df_trip_energy)


In [ ]:
results = {}
for scenario in params_set.columns:
    params = params_set[scenario].to_dict()
    print("")
    print(f"******************************")
    print(f"Running scenario: {scenario}")
    df_soc = sq.truck_soc(df_activities = df_occ_energy, **params)
    results_dict = sq.evaluate_charging_distribution(df_soc, evaluate_per_fleet=False, **params)
    results.update({scenario: results_dict})
    
df_results = pd.DataFrame.from_dict(results, orient='index')

In [ ]:
soc_results = {}
tours_energy_results = {}
for scenario in params_set.columns:
    params = params_set[scenario].to_dict()
    print("")
    print(f"******************************")
    print(f"Running scenario: {scenario}")
    df_soc = sq.truck_soc(df_activities = df_occ_energy, **params)
    results_dict = sq.evaluate_charging_distribution(df_soc, evaluate_per_fleet=False, **params)
    results.update({scenario: results_dict})
    
    # Neu: Berechne und speichere df_tours_energy pro Szenario
    df_tours_energy_scenario = dp.aggregate_tours(df_soc, energy=True)
    tours_energy_results[scenario] = df_tours_energy_scenario
    # Speichere df_soc pro Szenario
    soc_results[scenario] = df_soc

In [ ]:
df_results.plot(
    kind='bar',
    stacked=True,
    color=[line_colors['TUMBlue1'], line_colors['Purple'], line_colors['TUMOrange']],
    figsize=(8, 4)
)
plt.ylabel('Proportion')
plt.xlabel('Scenario')
plt.title('Stacked Barplot of Charging Location Proportions per Scenario')
plt.legend(title='Location')
plt.tight_layout()
plt.show()
df_results.to_csv('output/csvs/charging_distribution.csv')

In [ ]:
params = params_set[DEFAULT_SCENARIO].to_dict()

threshold = params['batt_cap'] * ( params['soc_max'] - params['soc_min'] )

### Tour statistics

In [ ]:
"""
In contrast to df_tours, df_tours_energy also includes non-driving activities between tracks. 
Thus, some additional parameters are included

    Stop time is the stop time of the last track equivalently to df_tours, 
    End time additionally includes the time spent at the home base after the tour's last track (i.e. = start time of the next tour)
   
    driving_duration is the sum of the duration of all driving activities in the tour, equivalently to duration in df_tours
    duration is the time between the start of the tour's first track and the end of its last activity 
     (i.e the duration between the start of the tour and the start of the next tour)
"""

df_tours_energy = dp.aggregate_tours(soc_results[DEFAULT_SCENARIO], energy=True)
df_tours_energy

In [ ]:
# -------------------------------------------------------------------------------------------
#                           SOC BY FREIGHT FORWARDER VIOLIN PLOTS (ENERGY)
# -------------------------------------------------------------------------------------------

def plot_soc_min_violin(tours_energy_results, cut=-1):
    """
    Create a single violin plot showing the distribution of soc_no_public_charging_min for each freight forwarder.
    Overlays three scenarios per forwarder: 50kW (light transparent), 150kW (solid), 350kW (medium transparent).
    """
    # Extract and prepare data for the three scenarios
    scenarios = ['low_home', 'default', 'high_home']  # Adjusted order to plot high_home last for visibility of its lines
    dfs = []
    for sc in scenarios:
        if sc in tours_energy_results:
            df_sc = tours_energy_results[sc].copy()
            df_sc['scenario'] = sc  # Add new column for scenario
            dfs.append(df_sc)
    
    if not dfs:
        raise ValueError("No matching scenarios found in tours_energy_results.")
    
    plot_df = pd.concat(dfs, ignore_index=True)
    
    # Adjust colors: Based on palette_ff, but with transparency
    # Create a custom palette with alpha values
    alpha_map = {'low_home': 0.3, 'default': 1.0, 'high_home': 0.6}
    unique_ff = sorted(plot_df['freight_forwarder'].unique())
    palette_adjusted = {}
    for ff in unique_ff:
        base_color = palette_ff[ff]  # Assume palette_ff is a dict with colors per FF
        palette_adjusted[ff] = {}  # Nested for scenarios
        for sc in scenarios:
            rgba = list(matplotlib.colors.to_rgba(base_color))
            rgba[3] = alpha_map.get(sc, 1.0) 
            palette_adjusted[ff][sc] = tuple(rgba)
    
    # Since sns.violinplot does not directly support nested palettes, plot per scenario separately and overlay
    plt.figure(figsize=(textwidth, h_169))
    for sc in scenarios:
        df_sc = plot_df[plot_df['scenario'] == sc]
        num_col_before = len(plt.gca().collections)
        sns.violinplot(
            data=df_sc,
            x='freight_forwarder',
            y='soc_no_public_charging_min',
            hue='freight_forwarder',  # Hue on FF to maintain colors per FF
            palette={ff: palette_adjusted[ff][sc] for ff in unique_ff},
            inner='quartile' if sc == 'default' else None,  # Show horizontal lines only for default (150kW)
            cut=cut,
            linewidth=1.5 if sc == 'default' else 1.0,  # Thicker line for default
            bw_adjust=0.5,
            dodge=False,  # Overlay instead of side-by-side
            legend=False,
            alpha=alpha_map.get(sc, 1.0)  # Alpha for the fill (combined with color)
        )
        if sc == 'low_home':
            for i in range(num_col_before, len(plt.gca().collections)):
                plt.gca().collections[i].set_hatch('//')
                plt.gca().collections[i].set_edgecolor('black')
                plt.gca().collections[i].set_linewidth(0.5)
        if sc == 'high_home':
            for i in range(num_col_before, len(plt.gca().collections)):
                plt.gca().collections[i].set_linestyle('--')  # Make outer lines dashed for 350kW
                plt.gca().collections[i].set_linewidth(1.0)
    
    plt.grid(True, axis='y', linestyle='--', linewidth=0.5, color='lightgrey', alpha=0.7)
    plt.xlabel('Freight Forwarder')
    plt.ylabel('Minimum State of Charge (SoC) / \%')
    plt.ylim(cut, 1)
    plt.gca().yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    # plt.title('Distribution of Minimum SoC per Tour by Charging Power')
    plt.xticks(
        ticks=range(len(unique_ff)),
        labels=unique_ff
    )
    
    # Add dashed line at the threshold (assuming y=0 as the physical threshold for SoC, to indicate clipped area and outliers)
    plt.axhline(y=0, linestyle='--', color='black', alpha=0.7, linewidth=1.2, label='0% SoC Threshold (clipped below)')
    
    # Add legend at the bottom, horizontal and flat
    # Use a neutral color for legend patches to represent structure for all freight forwarders
    legend_colors = {
        'low_home': 'gray',  # Darker gray for better visibility
        'default': 'darkgray',  # Strong, solid color
        'high_home': 'lightgray'  # Distinct and visible
    }
    power_map = {'low_home': '50', 'default': '150', 'high_home': '350'}
    handles = [
        mpatches.Patch(color=legend_colors[sc], alpha=min(1.0, alpha_map[sc] + 0.2), hatch='//' if sc == 'low_home' else None, label=f'{power_map[sc]} kW') 
        if sc != 'high_home' else
        mlines.Line2D([], [], color=legend_colors[sc], alpha=min(1.0, alpha_map[sc] + 0.2), linestyle='--', linewidth=2, label=f'{power_map[sc]} kW')
        for sc in scenarios
    ]
    plt.legend(handles=handles, title='Home Charging Power', loc='lower center', bbox_to_anchor=(0.5, -0.02),
               ncol=3, borderaxespad=0., frameon=False)
    
    plt.tight_layout()

plot_soc_min_violin(tours_energy_results, cut=-1)

## Charging Energy Demand per Home Base

In [ ]:
daily_demands = dp.daily_energy_demands(df_tours_energy, threshold, params['charging_powers'])

print(daily_demands.head())

# Most critical for ff 4. Where we have a representative sample of 10% of all of the ff's trucks --> daily load profiles would be 10x of these results
# 90/135 or 66 % of all recorded days could not be served with one traffo even if we assume an even load profile throughout the day 
# (max recharge in a day with 1 traffo = 630 kW * 24h = 15120 kWh)

In [ ]:

# ------------------------------------------------------------------------------
#                             WEEKLY ENERGY DEMAND BOXPLOT
# ------------------------------------------------------------------------------


def plot_weekly_energy_demand_boxplot(df_loads):
    """
    Creates a figure with a central boxplot showing all home bases' energy demand together
    at the top, and individual plots for each home base (CID) below.
    
    Parameters:plot_weekly_energy_demand_boxplot(daily_demands)
    -----------
    df_loads : DataFrame
        DataFrame containing daily energy demand data with day, energy_demand_kwh, cid, and freight_forwarder columns
    """
    
    # Convert day column to datetime and extract weekday
    df_loads['day'] = pd.to_datetime(df_loads['day'])
    df_loads['weekday'] = df_loads['day'].dt.weekday  # 0 = Monday, 6 = Sunday

    # Create subplots with a 3 column x 5 row grid (15 subplots total)
    fig = plt.figure(figsize=(textwidth, h_169))
    
    # Add a GridSpec with 5 rows and 3 columns
    gs = fig.add_gridspec(5, 3, height_ratios=[2, 1, 1, 1, 1])
    
    # Add the central plot at the top, spanning all columns
    central_ax = fig.add_subplot(gs[0, :])
    
    # Create the individual CID axes
    axes = []
    for i in range(1, 5):  # Rows 1-4
        for j in range(3):  # 3 columns
            ax = fig.add_subplot(gs[i, j])
            axes.append(ax)
    
    # Get unique CIDs
    cids = sorted(df_loads['cid'].unique())
    
    # Prepare data for the central plot
    all_boxplot_data = []
    all_weekday_labels = []
    all_weekday_numbers = []  # Store the actual weekday numbers
    
    for weekday in range(7):
        weekday_data = df_loads[df_loads['weekday'] == weekday]['energy_demand_kwh'].tolist()
        if len(weekday_data) > 0:  # Only add if there's data for this weekday
            all_boxplot_data.append(weekday_data)
            all_weekday_labels.append(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'][weekday])
            all_weekday_numbers.append(weekday)
    
    # Add the average data for the central plot (all weekdays combined)
    all_avg_data = df_loads['energy_demand_kwh'].tolist()
    all_boxplot_data.append(all_avg_data)
    all_weekday_labels.append('Combined')
    all_weekday_numbers.append('Combined')
    
    # Create central boxplot with all data
    scaling_factor = textwidth / 16
    bp_central = central_ax.boxplot(
        all_boxplot_data,
        patch_artist=True,
        showmeans=True,
        meanline=True,
        meanprops=dict(color=colors['TUMGreen3'], linestyle='--', linewidth=1.5 * scaling_factor),
        medianprops=dict(color=colors['TUMOrange'], linewidth=1.5 * scaling_factor),
        showfliers=True,
        flierprops=dict(marker='o', color='red', alpha=0.5, markersize=3),
        whiskerprops=dict(color=colors['TUMBlack'], linewidth=0.8 * scaling_factor),
        capprops=dict(color=colors['TUMBlack'], linewidth=0.8 * scaling_factor),
        widths=0.55 * scaling_factor,
        positions=range(len(all_boxplot_data)),
        labels=all_weekday_labels
    )
    
    # Style central boxplot
    central_ax.set_title('All Home Bases Combined')
    central_ax.tick_params(axis='x', )
    central_ax.tick_params(axis='y', )
    central_ax.grid(True, which='both', axis='y', linestyle='--', linewidth=0.5, color='lightgrey', alpha=0.7)
    central_ax.set_ylim(bottom=0)
    
    # Define legend handles directly in the central plot
    mean_line = mlines.Line2D([], [], color=colors['TUMGreen3'], linestyle='--', label='Mean', linewidth=3 * scaling_factor)
    median_line = mlines.Line2D([], [], color=colors['TUMOrange'], label='Median', linewidth=3 * scaling_factor)
    # Add the legend to the top left corner of the central plot
    central_ax.legend(handles=[median_line, mean_line], loc='upper left',  frameon=True, ncol=2)
    
    # Assign colors to the central boxplots based on weekday or 'Combined'
    for i, (box, weekday_or_combined) in enumerate(zip(bp_central['boxes'], all_weekday_numbers)):
        box.set_facecolor(palette_weekdays[weekday_or_combined])
        # Add a slightly thicker border to the average box to make it stand out
        if weekday_or_combined == 'Combined':  # Average box
            box.set(linewidth=2.0 * scaling_factor)
    
    # Annotate central plot median values
    for n, (median_feature, demands) in enumerate(zip(bp_central['medians'], all_boxplot_data)):
        demands_series = pd.Series(demands)
        median_value = demands_series.median()
        x_median, y_median = median_feature.get_xydata()[1]
        central_ax.text(x_median, y_median, f'{median_value:.2f}', 
                horizontalalignment='center', color=colors['TUMBlack'],
                bbox=dict(boxstyle='round', pad=0.3 * scaling_factor, facecolor=colors['TUMWhite'], 
                        edgecolor=colors['TUMOrange'], alpha=0.9))
    
    # Process each CID for individual plots
    for i, cid in enumerate(cids):
        if i >= len(axes):
            break
            
        # Filter data for current CID
        cid_data = df_loads[df_loads['cid'] == cid]
        
        # Get the freight forwarder for this CID
        ff = cid_data['freight_forwarder'].iloc[0] if not cid_data.empty else "Unknown"
        
        # Calculate and print daily energy demand per home base
        print(f"\nHome Base CID {cid} - Daily Energy Demand:")
        weekday_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
        daily_stats = cid_data.groupby('weekday')['energy_demand_kwh'].agg(['mean', 'std', 'median', 'count']).reset_index()
        daily_stats['weekday_name'] = daily_stats['weekday'].apply(lambda x: weekday_names[x])
        print(daily_stats[['weekday_name', 'mean', 'std', 'median', 'count']])
        print(f"Overall mean: {cid_data['energy_demand_kwh'].mean():.2f} kWh")
        
        # Prepare boxplot data with only available weekdays
        boxplot_data = []
        weekday_labels = []
        weekday_numbers = []  # Store the actual weekday numbers
        
        for weekday in range(7):
            weekday_data = cid_data[cid_data['weekday'] == weekday]['energy_demand_kwh'].tolist()
            if len(weekday_data) > 0:  # Only add if there's data for this weekday
                boxplot_data.append(weekday_data)
                weekday_labels.append(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'][weekday])
                weekday_numbers.append(weekday)
        
        # Add the average data for this CID (all weekdays combined)
        avg_data = cid_data['energy_demand_kwh'].tolist()
        boxplot_data.append(avg_data)
        weekday_labels.append('Combined')
        weekday_numbers.append('Combined')
        
        # Create boxplot
        bp = axes[i].boxplot(
            boxplot_data,
            patch_artist=True,
            showmeans=True,
            meanline=True,
            meanprops=dict(color=colors['TUMGreen3'], linestyle='--', linewidth=1.5 * scaling_factor),
            medianprops=dict(color=colors['TUMOrange'], linewidth=1.5 * scaling_factor),
            showfliers=True,
            flierprops=dict(marker='o', color='red', alpha=0.5, markersize=3),
            whiskerprops=dict(color=colors['TUMBlack'], linewidth=0.8 * scaling_factor),
            capprops=dict(color=colors['TUMBlack'], linewidth=0.8 * scaling_factor),
            widths=0.55 * scaling_factor,
            positions=range(len(boxplot_data)),
            labels=weekday_labels
        )
        
        # Set x-axis and y-axis tick labels font size
        axes[i].tick_params(axis='x', labelsize=10 * scaling_factor)
        axes[i].tick_params(axis='y', labelsize=10 * scaling_factor)
        
        # Annotate median values
        for n, (median_feature, demands) in enumerate(zip(bp['medians'], boxplot_data)):
            demands_series = pd.Series(demands)
            median_value = demands_series.median()
            x_median, y_median = median_feature.get_xydata()[1]
            axes[i].text(x_median, y_median, f'{median_value:.2f}', 
                    horizontalalignment='center', color=colors['TUMBlack'],
                    bbox=dict(boxstyle='round', pad=0.3 * scaling_factor, facecolor=colors['TUMWhite'], 
                            edgecolor=colors['TUMOrange'], alpha=0.9))
        
        # Assign colors to the boxplots based on weekday or 'Combined'
        for j, (box, weekday_or_combined) in enumerate(zip(bp['boxes'], weekday_numbers)):
            box.set_facecolor(palette_weekdays[weekday_or_combined])
            # Add a slightly thicker border to the average box to make it stand out
            if weekday_or_combined == 'Combined':  # Average box
                box.set(linewidth=2.0 * scaling_factor)
        
        # Set title for each subplot with both freight forwarder and CID
        axes[i].set_title(f'Freight Forwarder {ff} - Home Base CID {cid}')
        
        # Add grid
        axes[i].grid(True, which='both', axis='y', linestyle='--', linewidth=0.5, color='lightgrey', alpha=0.7)
        
        # Set y-axis limit
        axes[i].set_ylim(bottom=0)
        
        # Customize spines
        for spine in axes[i].spines.values():
            spine.set_edgecolor(colors['TUMBlack'])
            spine.set_linewidth(0.8 * scaling_factor)
    
    # Remove any empty subplots if there are fewer than 14 CIDs
    for i in range(len(cids), len(axes)):
        fig.delaxes(axes[i])
    
    # Add common labels
    fig.text(0.01, 0.5, 'Energy Demand / kWh', ha='center', va='center', rotation='vertical')

    plt.tight_layout(rect=[0.02, 0.03, 1, 0.95])  # Adjust layout to make room for common labels
    # plt.savefig('output/figures/energy/home_base_weekday_energy_boxplots.pdf', bbox_inches='tight')
    # plt.show()
    
# plot_weekly_energy_demand_boxplot(daily_demands)

## Create Load Profiles

In [ ]:
df_charging_stats = pd.DataFrame()
df_load_profiles = pd.DataFrame()
# Call the function with the load threshold parameter (default is 630 kW)
for scenario in params_set.columns:
    params = params_set[scenario].to_dict()
    print(f"Calculating load profiles for scenario: {scenario}")
    lp_, charging_stats = lp.calculate_charging_load_profiles(
        soc_results[scenario], charging_power=params['charging_powers']['home base'], load_threshold=630, charging_strategy='immediate')
    
    lp_['scenario'] = scenario
    df_load_profiles = pd.concat([df_load_profiles, lp_.reset_index()], axis=0)

    df_stats = pd.DataFrame.from_dict(charging_stats)
    df_stats['scenario'] = scenario
    df_charging_stats = pd.concat([df_charging_stats, df_stats.reset_index(names='cid')], axis=0)
    
display(df_charging_stats)
display(df_load_profiles)

In [ ]:
# ------------------------------------------------------------------------------
#                             LOAD PROFILE PLOT
# ------------------------------------------------------------------------------
# Hardcoded fleet sizes based on the provided image information for 6 freight forwarders

# Selected CIDs
selected_cids = [412, 842, 809]
# selected_cids = df_charging_stats.cid.unique()

palette_grid_load = {
    'mean': colors['TUMBlue1'],
    'max': colors['TUMOrange'],
    'threshold': colors['TUMGreen3'],
    'threshold2': colors['TUMGreen2'],
}

linestyles_scenarios = {'default': '-', 
    'low_home': '--', 
    'high_home': ':', 
    'destination': '-.', 
    'big_batt': (0, (3, 1, 1, 1)),
    'ultra_home': (0, (5, 2)),
    'backup1': (0, (1, 1)),
    'backup2': (0, (4, 1, 2, 1))
    }

def plot_load_profiles_grid_paper(load_profiles, selected_cids):
    _, axes = plt.subplots(len(selected_cids), 1, figsize=(textwidth, 2 * h_169), sharex=True)
    scenarios = load_profiles['scenario'].unique()    
    for scenario in scenarios:
        if scenario == 'ultra_home':
            continue
        for ax, cid in zip(axes, selected_cids):
            df = load_profiles.loc[load_profiles.scenario==scenario,[cid,]]
            df['date'] = df.index.date
            df['15min'] = df.index.floor('15min')

            date_nonzero_energy = df.groupby('date')[cid].sum().gt(0)
            df = df.loc[df['date'].isin(date_nonzero_energy[date_nonzero_energy].index)]
 
            df_15min_load = df.groupby('15min')[cid].mean()
            plot_data = df_15min_load.groupby(df_15min_load.index.time).agg(['min', 'mean', 'max', 'median'])
            plot_data['hour'] = plot_data.index.map(lambda t: t.hour + t.minute / 60 + t.second / 3600)
           
            ax.step(x=plot_data.hour, y=plot_data['mean'], label=f'{scenario_names[scenario]}', lw=1.2, color=palette_grid_load['mean'],
                    linestyle=linestyles_scenarios[scenario])

            ax.axhline(630, color=palette_grid_load['threshold'], lw=1)
            ax.axhline(1260, color=palette_grid_load['threshold2'], ls='--', lw=1)

            ax.set_title(f'CID {cid}')
            ax.set_xticks(range(0, 25, 6))
            ax.xaxis.set_minor_locator(mticker.FixedLocator(range(0, 25, 1)))
            ax.set_xticklabels(['00:00', '06:00', '12:00', '18:00', '24:00'])
            ax.set_xlim(0,24)
            ax.grid('on')
            
    plt.suptitle('Daily Load Profiles', fontweight='bold', y=0.98)
    plt.legend(loc='upper center', title='Scenario', bbox_to_anchor=(0.5, -.23), ncol=4)
    return plot_data
print("Selected CIDs for load profile plot:", selected_cids)
# Call the function with all load profiles at once
df = plot_load_profiles_grid_paper(df_load_profiles.set_index('time_'), selected_cids)
